# GPU Profiling with rocprof (ROCProfiler-SDK)

## Abstract

<p style="text-align:justify;">In this lab, you will profile a HIP GPU workload using rocprofv3, the ROCProfiler-SDK command line tool. You will collect runtime traces and kernel execution statistics, generate Perfetto timelines, and analyze profiling data using CSV files and an optional ROCPD (SQLite) database. This lab demonstrates a trace-based profiling workflow recommended for GPU profiling and performance analysis.</p>

This lab should take approximately 60 minutes.

## Objectives

After completing this lab, you will be able to:

- Prepare and execute a HIP GPU workload for profiling by locating, building, and running the rocprofv3 example application on a GPU-based system
- Apply rocprofv3 to collect trace-based profiling data, including runtime execution flow and GPU kernel execution behavior
- Analyze GPU kernel performance and identify hotspots by interpreting kernel execution statistics and detailed dispatch records generated in CSV format
- Visualize and correlate CPU and GPU execution behavior using Perfetto timelines and optionally validate kernel execution time through offline analysis with the ROCPD (SQLite) database

## Introduction

<p style="text-align:justify;">The AMD ROCm software platform provides a comprehensive open-source ecosystem for developing and optimizing GPU applications on AMD accelerators. As modern GPU workloads become more complex, understanding execution behavior and identifying performance bottlenecks is critical for achieving high performance and efficient resource utilization. ROCm addresses this need through profiling tools, such as ROCProfiler-SDK (rocprofv3), which enable detailed, trace-based analysis of GPU kernel execution, runtime activity, and memory behavior.</p>

<p style="text-align:justify;">Profiling on GPU-based systems focuses on capturing execution timelines that reveal how CPU and GPU activities interact. By analyzing kernel execution, memory transfers, and synchronization events, developers can identify inefficiencies such as idle gaps, delays, and runtime overheads. This trace-based approach provides visibility into both kernel-level performance and system-level behavior, enabling a deeper understanding of how applications utilize GPU resources.</p>

### ROCm Profiling Workflow Overview

This lab illustrates a complete ROCm profiling workflow using a HIP-based GPU workload demonstrated with a matrix multiplication design. The workflow consists of three main stages:

**Stage 1: HIP Application (GPU Workload)** - The starting point is a HIP application that executes GPU kernels and performs memory operations. This application serves as the target workload for profiling.

**Stage 2: HIP Runtime & ROCm Runtime** - When the application runs, the HIP runtime and ROCm runtime manage GPU resource allocation, kernel dispatches, memory transfers, and synchronization. These runtime layers coordinate the interaction between CPU and GPU.

**Stage 3: ROCProfiler-SDK (rocprofv3)** - The rocprofv3 profiling tool intercepts and records runtime activity, capturing three key categories of data: runtime tracing (CPU-GPU execution flow and API calls), kernel dispatch tracing (individual kernel launches with timing), and memory operations (data transfers and allocations).

**Stage 4: Profiling Outputs** - The profiler generates multiple output formats for analysis: CSV summaries containing kernel statistics and trace data, Perfetto trace files (.pftrace) for timeline visualization, and optionally ROCPD SQLite database for advanced offline analysis.

**Stage 5: Analysis & Visualization** - Using these outputs, developers can perform kernel hotspot analysis to identify performance-critical kernels, CPU-GPU timeline correlation to understand execution dependencies, and bottleneck identification to uncover optimization opportunities.

<p style="text-align:justify;">The lab begins with environment readiness and workload validation, where you will verify the availability of the ROCm profiling tools and HIP compiler, build a sample application, and execute it to confirm correct GPU operation. This establishes a reliable baseline before profiling. The second part focuses on profiling using rocprofv3, where you will collect runtime traces to capture the CPU-GPU execution flow and perform kernel-level profiling to generate execution statistics. In the final part, you will analyze the collected profiling data using CSV summaries, Perfetto timelines, and optional ROCPD database queries, enabling correlation of CPU activity with GPU execution.</p>

After completing this lab, you will gain practical experience with the ROCm profiling tools, learn how to interpret trace-based performance data, and develop the skills needed to identify bottlenecks and iteratively optimize GPU applications.

<p style="text-align:justify;">**Note on GPU Architecture Compatibility**: This lab uses profiling tools and examples that are compatible with AMD GPUs. While the tutorial content and profiling methodology apply broadly, ensure that your compilation targets match your physical GPU architecture using the appropriate `--offload-arch` flag when building HIP applications.</p>

## Step 1: Readying the Environment and Setting Up the Workload

<p style="text-align:justify;">The first step is to verify that the ROCm profiling toolchain and HIP compiler are properly installed and accessible. You will then locate the rocprofv3 example workload, build it, and run it to confirm correct GPU execution before profiling. This establishes a reliable baseline and ensures that the application executes successfully on your GPU hardware.</p>

### Verify the Environment

First, verify the availability of rocprofv3, the profiling tool that will be used throughout this lab. The `which` command checks whether the tool is installed and accessible through the system PATH.

In [1]:
!which rocprofv3

/opt/rocm/bin/rocprofv3


If available, the command will display the installation path of rocprofv3. Next, verify the availability of the HIP compiler, which is required to build GPU workloads.

In [2]:
!which hipcc

/opt/rocm/bin/hipcc


If available, the command will display the installation path of hipcc. Now clone the ROCm examples repository, which contains the profiling example application for this lab.

In [3]:
!git clone https://github.com/ROCm/rocm-examples.git

Cloning into 'rocm-examples'...
remote: Enumerating objects: 37267, done.
remote: Counting objects: 100% (6177/6177), done.
remote: Compressing objects: 100% (1585/1585), done.
remote: Total 37267 (delta 5092), reused 4670 (delta 4591), pack-reused 31090 (from 2)
Receiving objects: 100% (37267/37267), 46.63 MiB | 77.27 MiB/s, done.
Resolving deltas: 100% (30975/30975), done.


This clones the ROCm examples repository into the current directory. The repository contains various HIP programming examples and tools demonstrations, including the rocprofv3 profiling example. Navigate to the rocprofv3 example directory, which contains the HIP matrix multiplication application that will be profiled in this lab.

In [4]:
%cd rocm-examples/Tools/rocprofv3

/workspace/rocm-examples/Tools/rocprofv3


This changes the working directory to the rocprofv3 example location. Before building, clean any previous build artifacts to ensure a clean and reproducible build.

This removes any previously compiled binaries and intermediate files. Now build the profiling workload using the provided Makefile.

In [5]:
!make

/opt/rocm/bin/hipcc -std=c++17 -Wall -Wextra -I ../../Common -I ../../External  -o rocprofv3-matmul main.hip 
/opt/rocm/bin/hipcc -std=c++17 -Wall -Wextra -DEXAMPLE_ROCTX -I ../../Common -I ../../External  -o rocprofv3-matmul-roctx main.hip  -lrocprofiler-sdk-roctx


This compiles the HIP matrix multiplication application and generates the `rocprofv3-matmul` executable. The compilation process uses `hipcc` to compile HIP source code and link it with the necessary ROCm runtime libraries. Before profiling, run the application normally to validate functional correctness.

In [6]:
!./rocprofv3-matmul

Matrix multiplication: [2048x1024] * [1024x1024], block size: 16x16
Validation passed.


<p style="text-align:justify;">This executes the application without profiling to confirm that it runs successfully on your GPU hardware. Once successful, you should see a **Validation Passed** message, indicating that the matrix multiplication computed on the GPU matches the expected results. This confirms that the GPU execution is working correctly and the application is ready for profiling.</p>

## Step 2: Profiling the GPU Using rocprofv3

<p style="text-align:justify;">With the environment verified and the application validated, you will now run the application under rocprofv3 to collect trace-based profiling data. This step captures GPU kernel execution behavior, runtime activity, and memory operations during application execution. The profiling process involves two main activities: runtime trace profiling to capture the CPU-GPU execution flow, and kernel trace profiling to collect detailed kernel execution statistics.</p>

### Perform Runtime Trace Profiling

Runtime trace profiling captures the execution flow between the CPU and GPU, including HIP runtime API calls, kernel dispatches, memory copy operations, and synchronization events. This provides a timeline view of how the application orchestrates GPU work. The `--runtime-trace` option enables this tracing mode.

In [7]:
!rocprofv3 --runtime-trace -- ./rocprofv3-matmul

W20261023 19:49:13.061946 140266496448064 simple_timer.cpp:55] [rocprofv3] tool initialization ::     0.009012 sec
W20261023 19:49:13.062159 140266496448064 simple_timer.cpp:55] [rocprofv3] './rocprofv3-matmul' ::     0.000000 sec
W20261023 19:49:13.068169 140266496448064 tool.cpp:2422] HIP (runtime) version 7.2.0 initialized (instance=0)
W20261023 19:49:13.127019 140266496448064 tool.cpp:2422] HSA version 8.20.3 initialized (instance=0)
Matrix multiplication: [2048x1024] * [1024x1024], block size: 16x16
Validation passed.
W20261023 19:49:13.266342 140266496448064 simple_timer.cpp:55] [rocprofv3] './rocprofv3-matmul' ::     0.204184 sec
W20261023 19:49:13.288023 140266496448064 generateRocpd.cpp:582] writing SQL database for process 386 on node 949270026
E20261023 19:49:13.288478 140266496448064 generateRocpd.cpp:605] Opened result file: /workspace/rocm-examples/Tools/rocprofv3/jupyter-user-261023194507-ad6b5ed8/386_results.db (UUID=00020683-4e54-7e54-b36a-47f22d5a52bc)
W20261023 19:49

<p style="text-align:justify;">This command runs the application under rocprofv3 with runtime tracing enabled. The profiler intercepts and records HIP runtime activity, GPU kernel dispatches, and memory copy operations while the application executes normally. The double dash (`--`) separates rocprofv3 options from the application command.</p>

<p style="text-align:justify;">The output shows rocprofv3 initializing and attaching to the ROCm runtime, followed by normal execution of the matrix multiplication workload. After execution completes, profiling data is written to structured outputs in a directory named after the system hostname. The output includes a ROCPD database (`.results.db` file), CSV files for statistics and traces, and Perfetto trace files. The profiler confirms that CPU-GPU execution flow and memory activity have been captured successfully.</p>

<p style="text-align:justify;">Observe the hostname in the output path: `/home/<username>/rocm-examples/Tools/rocprofv3/<hostname>/xxx.results.db`. The profiling outputs are organized in a directory named after the hostname, making it easy to identify profiling sessions from different systems. Change into this output directory to examine the generated profiling artifacts.</p>

In [8]:
# Replace <hostname> with your actual hostname from the output above
%cd <hostname>

[Errno 2] No such file or directory: '<hostname>'
/workspace/rocm-examples/Tools/rocprofv3


List the profiling artifacts to confirm that runtime trace data has been generated and stored.

In [9]:
!ls

CMakeLists.txt			    rocprofv3-advanced.sh
Makefile			    rocprofv3-basic.sh
README.md			    rocprofv3-matmul
jupyter-user-261023194507-ad6b5ed8  rocprofv3-matmul-roctx
kernel_filter.yml		    wavefront_stats.yml
main.hip


The listed files confirm that profiling data from the runtime tracing step has been generated. You will see multiple files including runtime statistics CSV files, kernel statistics CSV files, and the ROCPD database file with a `.results.db` extension. These files contain the captured profiling data in different formats for various types of analysis.

Return to the build directory to prepare for the next profiling step.

In [10]:
%cd ..

/workspace/rocm-examples/Tools


This ensures that subsequent profiling commands are run from the directory containing the application binary.

### Perform Kernel Trace and Hotspot Profiling

<p style="text-align:justify;">Kernel trace profiling focuses specifically on GPU kernel execution behavior, recording detailed information about each kernel dispatch including execution time, launch parameters, and correlation with host-side activity. The `--kernel-trace` option enables per-dispatch kernel tracing, while the `--stats` option generates aggregated kernel execution statistics. These combined options provide both summary-level hotspot analysis and detailed per-launch execution data.</p>

Before collecting new kernel profiling data, remove any previous kernel profiling CSV files to avoid mixing old and new results.

In [11]:
!rm -f *_kernel_stats.csv *_kernel_trace.csv

This deletes kernel profiling CSV files from earlier runs, ensuring that the analysis in the next step reflects only the current profiling session. Now collect kernel execution statistics and detailed dispatch traces in CSV format.

In [12]:
!rocprofv3 --kernel-trace --stats --output-format csv -- ./rocprofv3-matmul

Traceback (most recent call last):
  File "/opt/rocm/bin/rocprofv3", line 1804, in <module>
    ec = main(sys.argv[1:])
         ^^^^^^^^^^^^^^^^^^
  File "/opt/rocm/bin/rocprofv3", line 1785, in main
    ec = run(app_args, args, pass_id=pass_idx)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/rocm/bin/rocprofv3", line 1713, in run
    os.execvpe(app_args[0], app_args, env=app_env)
  File "<frozen os>", line 618, in execvpe
  File "<frozen os>", line 632, in _execvpe
FileNotFoundError: [Errno 2] No such file or directory: './rocprofv3-matmul'


<p style="text-align:justify;">This command profiles GPU kernel execution behavior and records both aggregated statistics and per-dispatch details in CSV files. The `--output-format csv` option explicitly requests CSV output format, which is easy to parse and analyze using command-line tools or spreadsheet applications.</p>

<p style="text-align:justify;">Kernel execution data is captured and written to files such as `*_kernel_stats.csv` (aggregated statistics per kernel) and `*_kernel_trace.csv` (detailed per-dispatch records). The profiler confirms that kernel-level profiling completed successfully. These CSV files will be analyzed in the next step to identify performance hotspots and examine detailed kernel behavior.</p>

## Step 3: Analyzing and Visualizing the Profiling Data

<p style="text-align:justify;">With profiling data collected, you will now analyze the results to identify GPU kernel hotspots, inspect detailed execution behavior, and visualize CPU-GPU timelines. This step uses multiple analysis techniques: examining kernel statistics to find performance hotspots, reviewing detailed dispatch traces to understand individual kernel launches, visualizing execution timelines using Perfetto for CPU-GPU correlation, and optionally querying the ROCPD SQLite database for advanced offline analysis.</p>

### Analyze the Kernel Statistics

<p style="text-align:justify;">The kernel statistics CSV file contains aggregated performance data for each GPU kernel executed during the profiling session. This data identifies which kernels consume the most GPU time, helping you focus optimization efforts on the most impactful areas. Navigate to the hostname-based output directory where the profiling results are stored.</p>

In [ ]:
# Replace <hostname> with your actual hostname
%cd <hostname>

View the kernel hotspot statistics by loading the CSV file into a pandas DataFrame for easy analysis and visualization.

In [ ]:
import pandas as pd
import glob

# Find the kernel stats CSV file
stats_files = glob.glob('*_kernel_stats.csv')

if not stats_files:
    print("ERROR: No kernel stats CSV file found in current directory.")
    print("Make sure you:")
    print("  1. Ran the profiling command: rocprofv3 --kernel-trace --stats --output-format csv -- ./rocprofv3-matmul")
    print("  2. Changed to the hostname output directory (e.g., cd <hostname>)")
    print("\nCurrent directory contents:")
    !ls -la
else:
    # Read and display the kernel statistics
    df = pd.read_csv(stats_files[0])
    display(df)

This code loads the kernel statistics CSV file and displays it as a pandas DataFrame. The DataFrame provides a clean, tabular view of the data that is easy to read in Jupyter notebooks. Pandas automatically formats the columns and handles data types appropriately.

### Understanding Kernel Hotspot Statistics

The kernel statistics output contains several columns that provide comprehensive information about kernel execution behavior:

**Name**: The name of the GPU kernel executed. For example, `matrix_multiplication_kernel<16u>` indicates a matrix multiplication kernel where the template parameter `<16u>` typically represents a tile or block size used in the implementation.

**Calls**: The number of times the kernel was launched during application execution. This count helps identify how frequently a kernel is invoked and whether performance issues stem from individual kernel efficiency or from excessive launch overhead.

**TotalDurationNs**: The total GPU execution time spent in this kernel across all launches, measured in nanoseconds. This is the cumulative time spent executing this kernel and represents the total GPU cycles consumed. For example, a value of 2,615,987 ns means the kernel consumed approximately 2.6 milliseconds of GPU time in total.

**AverageNs**: The average execution time per kernel launch, calculated as TotalDurationNs ÷ Calls. This metric reveals the typical execution time for a single invocation of the kernel. When a kernel is launched only once, the average equals the total duration.

**Percentage**: The share of total kernel execution time consumed by this kernel. This percentage indicates the relative importance of each kernel for overall GPU performance. A kernel consuming 100% of GPU time is the sole performance hotspot, making it the primary target for optimization.

**MinNs**: The minimum observed execution time for the kernel across all launches. This represents the best-case performance for this kernel and can help identify performance variability.

**MaxNs**: The maximum observed execution time for the kernel across all launches. This represents the worst-case performance and helps identify performance outliers or variable execution behavior.

**StdDev**: The standard deviation of execution time across all kernel launches. This metric measures performance consistency. A standard deviation of 0.0 indicates no variation in execution time, which occurs when the kernel is launched only once or when all launches execute in exactly the same time.

<p style="text-align:justify;">Each row represents a distinct GPU kernel and shows how often it was launched, the total and average execution time, and the percentage of overall GPU time consumed. In a typical matrix multiplication example, the matrix multiplication kernel accounts for 100% of GPU execution time, identifying it as the primary performance hotspot and the sole target for optimization efforts. When multiple kernels are present, this table helps prioritize optimization work by revealing which kernels contribute most to total execution time.</p>

### View Detailed Kernel Dispatch Information

<p style="text-align:justify;">While kernel statistics provide aggregated performance data, the kernel trace CSV file contains detailed per-launch execution records. This data reveals how individual kernel dispatches behave, including timing, correlation with CPU activity, and execution context. View the detailed kernel dispatch information to examine individual kernel launches.</p>

In [ ]:
import pandas as pd
import glob

# Find the kernel trace CSV file
trace_files = glob.glob('*_kernel_trace.csv')

if not trace_files:
    print("ERROR: No kernel trace CSV file found in current directory.")
    print("Make sure you:")
    print("  1. Ran the profiling command: rocprofv3 --kernel-trace --stats --output-format csv -- ./rocprofv3-matmul")
    print("  2. Changed to the hostname output directory (e.g., cd <hostname>)")
    print("\nCurrent directory contents:")
    !ls -la
else:
    # Read and display the kernel trace (showing first 10 rows for readability)
    df = pd.read_csv(trace_files[0])
    display(df.head(10))

This code loads the kernel trace CSV file and displays the first 10 rows in a pandas DataFrame. The DataFrame format makes it easy to examine detailed per-launch execution records. Each row corresponds to an individual kernel launch, showing execution timing and context information.

### Understanding Kernel Dispatch Trace Columns

The kernel trace output contains numerous columns that describe each kernel dispatch in detail:

**Kind**: The type of runtime event recorded. `KERNEL_DISPATCH` indicates that this record represents a GPU kernel launch event captured by the ROCm runtime. Other event types might include memory operations or synchronization events in more complex profiling sessions.

**Agent_Id**: The GPU device executing the kernel. For example, `Agent 1` corresponds to the first detected GPU in the system. In multi-GPU systems, different agents represent different physical GPUs, allowing you to track which device executed which kernel.

**Queue_Id**: The hardware command queue used to submit the kernel. Multiple queues enable concurrent execution of independent operations. Queue identifiers help track work submission patterns and identify potential queue contention or underutilization.

**Stream_Id**: The HIP stream ID used for kernel dispatch. Stream 0 represents the default stream, which is synchronous with respect to other work on that stream. Non-zero stream IDs indicate asynchronous execution streams that can overlap with work in other streams.

**Thread_Id**: The host (CPU) thread ID that issued the kernel launch. This identifier is useful for correlating CPU-side activity with GPU execution, especially in multi-threaded applications where different CPU threads submit work to different streams.

**Dispatch_Id**: A sequential identifier for this kernel dispatch in the profiling session. This helps distinguish multiple launches of the same kernel and track execution order across different kernels.

**Kernel_Id**: A unique identifier assigned to the kernel instance by the profiler. This internal identifier is used to correlate dispatch events with kernel symbol information and other profiling data.

**Kernel_Name**: The name of the executed GPU kernel, including template parameters and full function signature. For example, `void matrix_multiplication_kernel<16u>(...)` indicates the kernel name, template parameter, and argument types. This allows precise identification of which kernel variant was executed.

**Correlation_Id**: An identifier linking this kernel dispatch with corresponding HIP API calls and other runtime events. This correlation enables end-to-end CPU-GPU tracing, allowing you to connect a kernel launch on the GPU back to the specific HIP API call that initiated it on the CPU.

**Start_Timestamp**: The time when the kernel began execution on the GPU, usually measured in nanoseconds relative to a profiling epoch. This timestamp marks the beginning of GPU computation for this kernel.

**End_Timestamp**: The time when kernel execution completed on the GPU. The difference between end and start timestamps gives the kernel duration. This timestamp marks when the GPU finished executing the kernel and the results became available.

<p style="text-align:justify;">These detailed records allow you to analyze individual kernel launches, identify timing variations, correlate GPU activity with CPU-side events, and understand the execution context for each kernel dispatch. When analyzing performance, look for patterns such as consistent execution times (indicating stable performance), variations that might indicate resource contention, or correlations between launch parameters and execution duration.</p>

Return to the build directory to prepare for visualization steps.

In [ ]:
%cd ..

### Visualize Using the Perfetto Timeline

While CSV files provide detailed numerical data, timeline visualization offers intuitive visual insight into CPU-GPU execution patterns, revealing overlaps, idle periods, and synchronization behavior. Perfetto is a powerful web-based trace viewer that displays runtime events on a timeline, making it easy to identify execution inefficiencies and understand temporal relationships between events.

Generate a Perfetto-compatible trace by running the application under rocprofv3 with the appropriate output format.

In [ ]:
!rocprofv3 --runtime-trace --output-format pftrace -- ./rocprofv3-matmul

This reruns the application and exports runtime profiling data in Perfetto trace format. The `--output-format pftrace` option instructs rocprofv3 to generate a `.pftrace` file instead of CSV format. A Perfetto trace file (`.pftrace`) is generated in the hostname-based output directory. This file contains timeline data representing CPU and GPU execution events in a format that the Perfetto viewer can render.

To view the timeline, you need to transfer the `.pftrace` file to your local system and open it in a web browser. The Perfetto viewer runs entirely in the browser and does not require any installation or upload to external servers.

**Transfer the Perfetto trace file to your local Windows system**: Open a new terminal on your local Windows system and use `scp` to copy the `.pftrace` file from the remote system. Replace `<remote_user>`, `<remote_host>`, and `<remote_report_path>` with your actual values, and specify a local destination directory.

```bash
scp <remote_user>@<remote_host>:<remote_report_path>/<PID>_results.pftrace <local_destination>
```

For example:

```bash
C:\training> scp user@192.168.1.36:/home/user/rocm-examples/Tools/rocprofv3/hostname/468129_results.pftrace .
```

The `<PID>` in the filename represents the process ID of the profiled application, making each trace file uniquely identifiable.

**Open the Perfetto web viewer**: Open a web browser and navigate to the following URL:

```
https://ui.perfetto.dev
```

The Perfetto UI is a client-side web application that processes trace files locally in your browser without uploading data to any server. Click **Open trace file** from the menu and browse to the location where you saved the `.pftrace` file. Select the `<PID>_results.pftrace` file and click **Open**.

This loads the generated trace file into the Perfetto viewer. The timeline displays CPU runtime activity, GPU kernel execution, memory operations, and idle gaps in a unified view. This visualization provides visual insight into execution order and overlap, showing how different activities interleave in time. You can zoom, pan, and click on individual events to see detailed timing information and correlate CPU-side API calls with GPU-side kernel execution.

**Understanding the Perfetto Timeline**: The timeline view shows multiple horizontal tracks representing different types of activity. CPU tracks show HIP runtime API calls and host-side operations, while GPU tracks show kernel execution and memory transfers. Gaps between events reveal idle time when the GPU is not performing computation. By examining the timeline, you can identify opportunities for optimization such as overlapping computation with memory transfers, reducing idle gaps between kernel launches, or improving CPU-GPU synchronization patterns.

Once you complete your review, close the Perfetto web browser. The trace file remains on your local system for future analysis if needed.

### [Optional] Analyze Using the ROCPD (SQLite) Database

For advanced users who prefer SQL-based analysis or need to perform custom queries, rocprofv3 also generates a ROCPD (ROCm Profile Data) SQLite database. This database contains structured profiling data that can be queried using standard SQL, enabling complex analysis, custom reporting, and integration with external analysis tools. This step is optional but demonstrates the flexibility of the ROCm profiling ecosystem.

Navigate to the hostname-based output directory where the profiling database is stored.

In [ ]:
# Replace <hostname> with your actual hostname
%cd <hostname>

List the profiling databases to confirm that the ROCPD database file was generated.

In [ ]:
!ls *_results.db

<p style="text-align:justify;">This lists all `.db` files in the output directory. You should see one or more database files with names like `<PID>_results.db`, where `<PID>` is the process ID of the profiled application. Open the ROCPD database using the `sqlite3` command-line tool for offline query-based analysis. Replace `<PID>` with the actual process ID from the filename.</p>

```bash
$ sqlite3 <PID>_results.db
```

This opens the profiling results database in an interactive SQLite shell. The SQLite prompt indicates that structured profiling data is available for querying. The database schema includes tables for kernel dispatches, kernel symbols, runtime API calls, memory operations, and more.

Execute a SQL query to report the total GPU execution time per kernel in milliseconds. This query joins the kernel dispatch table with the kernel symbol table and aggregates execution time by kernel name.

### Example SQL Query for Kernel Hotspot Analysis

```sql
sqlite> SELECT ks.display_name AS kernel_name,
           SUM(kd."end" - kd.start) / 1e6 AS total_time_ms
        FROM rocpd_kernel_dispatch kd
        JOIN rocpd_info_kernel_symbol ks
          ON kd.kernel_id = ks.id
        GROUP BY ks.display_name
        ORDER BY total_time_ms DESC;
```

This SQL query performs several operations:

- Selects the kernel display name from the kernel symbol table
- Calculates the total execution time by summing the difference between end and start timestamps for all dispatches of each kernel
- Converts the time from nanoseconds to milliseconds by dividing by 1e6
- Joins the dispatch records with kernel symbol information based on kernel ID
- Groups the results by kernel name to aggregate execution time per kernel
- Orders the results by total execution time in descending order to show the most time-consuming kernels first

The query output reports the total GPU execution time per kernel in milliseconds, confirming the kernel hotspot identified earlier in the CSV analysis. This demonstrates that the same information is available through both CSV files and SQL queries, allowing you to choose the analysis method that best fits your workflow.

Exit the SQLite shell when you are finished with the database analysis.

```bash
sqlite> .exit
```

This returns you to the normal command prompt. The database remains available for future analysis if needed.

## Summary

<p style="text-align:justify;">In this lab, you used rocprofv3 to profile a HIP GPU workload on a GPU-based system. You captured runtime execution traces that revealed the CPU-GPU execution flow including HIP runtime API calls, kernel dispatches, and memory operations. You identified GPU kernel hotspots using aggregated kernel statistics that showed total execution time, launch counts, and performance percentages for each kernel. You examined per-kernel dispatch behavior through detailed trace records containing timing information, correlation IDs, and execution context for individual kernel launches. Finally, you visualized execution timelines using Perfetto to correlate CPU and GPU activities and identify idle gaps and synchronization patterns.</p>

<p style="text-align:justify;">This lab illustrated a complete trace-based GPU profiling workflow suitable for analyzing and optimizing GPU applications. The techniques and tools demonstrated here form the foundation for iterative performance optimization, enabling you to identify bottlenecks, understand execution behavior, and make data-driven decisions about code improvements. By combining CSV-based statistical analysis, visual timeline correlation, and optional SQL-based queries, you now have a comprehensive toolkit for GPU performance analysis on AMD platforms.</p>